# Network Visualization Demo

Directed graph views for the midterm manure benchmark problem structures and reference-solution overlays.

Graphviz is optional. If SVG rendering is unavailable on macOS, run:

`brew install graphviz`

`pip install graphviz`


In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "src").exists():
    REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

REPO_ROOT


In [ ]:
from IPython.display import Markdown, SVG, display

from src.llm_problem_interpreter import build_state_from_semantic_plan
from src.midterm_benchmark import (
    build_midterm_manure_expected_plan,
    build_midterm_manure_q2_expected_plan,
    build_midterm_manure_q3_expected_plan,
    build_midterm_manure_q4_expected_plan,
)
from src.network_graph import build_problem_graph_spec, build_solution_graph_spec
from src.network_visualizer import render_graphviz, render_mermaid
from src.solver_results import SolverResults


In [ ]:
OUTPUT_DIR = REPO_ROOT / "midterm_outputs" / "network_graphs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def show_graph(spec, name):
    svg_path = OUTPUT_DIR / f"{name}.svg"
    try:
        rendered_path = render_graphviz(spec, str(svg_path))
        display(SVG(filename=rendered_path))
        return rendered_path
    except RuntimeError as exc:
        display(Markdown(f"Graphviz unavailable: `{exc}`"))
        display(Markdown("```mermaid\n" + render_mermaid(spec) + "\n```"))
        return None


def make_reference_results(objective, bids, flows, technologies=None):
    return SolverResults(
        solver_name="reference_fixture",
        solver_status="reference_fixture",
        termination_condition="reference_fixture",
        solver_time_seconds=0.0,
        success=True,
        objective_value=objective,
        bid_allocations=bids,
        transport_flows=flows,
        technology_extents=technologies or {},
    )


In [ ]:
q1_state = build_state_from_semantic_plan(build_midterm_manure_expected_plan("canonical"))
q2_state = build_state_from_semantic_plan(build_midterm_manure_q2_expected_plan("canonical"))
q3_state = build_state_from_semantic_plan(build_midterm_manure_q3_expected_plan("canonical"))
q4_state = build_state_from_semantic_plan(build_midterm_manure_q4_expected_plan("canonical"))

q1_results = make_reference_results(
    850.0,
    {"B_Dairy_EauClaire": 1000.0, "B_Menomonie": 500.0, "B_BlackRiverFalls": 500.0},
    {"EauClaire_to_Menomonie": 500.0, "EauClaire_to_BlackRiverFalls": 500.0},
)
q2_results = make_reference_results(
    650.0,
    {"B_Dairy_EauClaire": 500.0, "B_Menomonie": 0.0, "B_BlackRiverFalls": 500.0},
    {"EauClaire_to_Menomonie": 0.0, "EauClaire_to_BlackRiverFalls": 500.0},
)
q3_results = make_reference_results(
    1050.0,
    {"B_Dairy_EauClaire": 1000.0, "B_Menomonie": 500.0, "B_BlackRiverFalls": 500.0},
    {"EauClaire_to_Menomonie": 500.0, "EauClaire_to_BlackRiverFalls": 500.0},
)
q4_results = make_reference_results(
    5800.0,
    {"B_DF_DM": 1000.0, "B_CF_DM": 0.0, "B_SF_DM": 500.0, "B_DC_Compost": 50.0},
    {"DF_to_CF": 0.0, "DF_to_SF": 500.0, "DF_to_Composter": 500.0, "Composter_to_DC": 50.0},
    {"Composter": 500.0},
)


## Q1 Problem Graph


In [ ]:
show_graph(build_problem_graph_spec(q1_state), "q1_problem_graph")


## Q1 Solution Graph


In [ ]:
show_graph(build_solution_graph_spec(q1_state, q1_results), "q1_solution_graph")


## Q2 Solution Graph


In [ ]:
show_graph(build_solution_graph_spec(q2_state, q2_results), "q2_solution_graph")


## Q3 Solution Graph


In [ ]:
show_graph(build_solution_graph_spec(q3_state, q3_results), "q3_solution_graph")


## Q4 Problem Graph


In [ ]:
show_graph(build_problem_graph_spec(q4_state), "q4_problem_graph")


## Q4 Solution Graph


In [ ]:
show_graph(build_solution_graph_spec(q4_state, q4_results), "q4_solution_graph")
